# Notebook 7 - Allrounder Stats

**Goal**: Identify genuine allrounders in IPL history (players who qualified with 50+ balls faced AND 50+ balls bowled in the same season), then rank them using within-pool z-scores.

**Why within-pool z-scores?** The standard impact score z-scores are computed against all 8,988 player-match rows, which includes specialist batters and pure bowlers. That means Hardik Pandya's 8.5 economy looks 'average' when benchmarked against Bumrah, and his 145 SR looks average when benchmarked against specialist openers. The right question for an allrounders page is: relative to other allrounders, how good is this player at batting AND bowling?

**Two outputs**:
- `allrounder_scores.csv` - per player per season: batting SR z-score and bowling economy z-score within the allrounder pool for that season
- `allrounder_season_best.csv` - one row per season with the best combined-z allrounder that season

## 1. Imports

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.3f}'.format)

## 2. Load Data

I load the full dataset without any era filter. The allrounder scores need to cover all seasons from 2008 onward so the dashboard's 'All Time' toggle works correctly.

In [2]:
df_all    = pd.read_csv('../data/processed/deliveries.csv')
df_all    = df_all[df_all['super_over'] == False].copy()
legal_all = df_all[df_all['is_wide'] == False].copy()

print(f'Rows (no super overs): {len(df_all):,}')
print(f'Seasons covered: {df_all["season"].min()} - {df_all["season"].max()}')
print(f'Matches: {df_all["match_id"].nunique():,}')

Rows (no super overs): 295,557
Seasons covered: 2008 - 2026
Matches: 1,243


## 3. Allrounder Pool

**Qualification rule**: A player must have 50+ legal balls faced AND 50+ legal balls bowled in the **same season**. This is a per-season check, not a career total. It prevents players like Bumrah (who may accumulate 50 batting balls across 15 seasons) from qualifying.

**Z-scoring approach**: I compute batting SR and bowling economy z-scores within each season's allrounder pool separately. This means a player is compared only against other allrounders in the same season - not against specialist batters or bowlers.

In [3]:
# --- Step 1: Allrounder pool per season (50+ balls faced AND 50+ balls bowled, same season) ---
bat_season_ar = (
    legal_all.groupby(['season', 'batter'])
    .size()
    .reset_index(name='balls_faced')
    .rename(columns={'batter': 'player'})
    .query('balls_faced >= 50')
)

bowl_season_ar = (
    legal_all.groupby(['season', 'bowler'])
    .size()
    .reset_index(name='balls_bowled')
    .rename(columns={'bowler': 'player'})
    .query('balls_bowled >= 50')
)

# Inner merge: only (season, player) pairs qualifying in both departments
ar_pool = bat_season_ar.merge(bowl_season_ar, on=['season', 'player'])
print(f'Allrounder-qualified player-seasons: {len(ar_pool)}')
print(f'Unique players: {ar_pool["player"].nunique()}')
print(f'Seasons covered: {sorted(ar_pool["season"].unique())}')

Allrounder-qualified player-seasons: 384
Unique players: 128
Seasons covered: [np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]


In [4]:
# --- Step 2: Batting SR per allrounder per season ---
bat_runs_ar = (
    legal_all.groupby(['season', 'batter'])['batter_runs']
    .sum()
    .reset_index(name='runs')
    .rename(columns={'batter': 'player'})
)
ar_pool = ar_pool.merge(bat_runs_ar, on=['season', 'player'])
ar_pool['sr'] = ar_pool['runs'] / ar_pool['balls_faced'] * 100

# --- Step 3: Bowling economy per allrounder per season ---
# Runs conceded includes wide extras (charged to bowler) - use df_all not legal_all
bowl_runs_ar = (
    df_all.groupby(['season', 'bowler'])['total_runs']
    .sum()
    .reset_index(name='runs_conceded')
    .rename(columns={'bowler': 'player'})
)
ar_pool = ar_pool.merge(bowl_runs_ar, on=['season', 'player'])
ar_pool['economy'] = ar_pool['runs_conceded'] / (ar_pool['balls_bowled'] / 6)

print('Sample batting SR range:', ar_pool['sr'].min().round(1), '-', ar_pool['sr'].max().round(1))
print('Sample economy range:   ', ar_pool['economy'].min().round(1), '-', ar_pool['economy'].max().round(1))

Sample batting SR range: 69.8 - 216.7
Sample economy range:    5.5 - 12.3


In [5]:
# --- Step 4: Z-score within each season's allrounder pool ---
# Comparing each player against allrounder peers only (not specialist bowlers like Bumrah)
ar_pool['bat_z'] = ar_pool.groupby('season')['sr'].transform(
    lambda x: (x - x.mean()) / x.std() if x.std() > 0 else 0.0
)
# Negate: lower economy = better, so an economical bowler gets a positive z-score
ar_pool['bowl_z'] = ar_pool.groupby('season')['economy'].transform(
    lambda x: -(x - x.mean()) / x.std() if x.std() > 0 else 0.0
)
ar_pool['combined_z'] = ar_pool['bat_z'] + ar_pool['bowl_z']

# --- Step 5: Match count per player per season ---
match_counts_ar = (
    df_all.groupby(['season', 'batter'])['match_id']
    .nunique()
    .reset_index(name='matches')
    .rename(columns={'batter': 'player'})
)
ar_pool = ar_pool.merge(match_counts_ar, on=['season', 'player'], how='left')

print('Z-score distribution:')
print(ar_pool[['bat_z', 'bowl_z', 'combined_z']].describe().round(3))

Z-score distribution:
        bat_z  bowl_z  combined_z
count 384.000 384.000     384.000
mean    0.000   0.000       0.000
std     0.976   0.976       1.326
min    -2.192  -2.598      -3.905
25%    -0.710  -0.641      -0.916
50%    -0.099   0.104      -0.008
75%     0.604   0.760       0.873
max     2.906   2.350       3.642


In [6]:
# --- Step 5: Bowling regularity filter ---
# A genuine allrounder must bowl at least 6 balls per match on average across their career.
# Without this, players who hit the 50-ball threshold via a lucky handful of matches
# (and had a good economy in those few appearances) distort the z-score rankings.
#
# At bowl_per_match >= 6: Rashid (37.0), Jadeja (22.8), Axar (22.2), Hardik (14.0),
# Russell (12.6), Maxwell (7.4), Stoinis (6.4) all pass. Low-volume bowlers like
# Parag and Vipraj Nigam are handled by the callback's per-window consistency filter.
bpm_career = (
    ar_pool.groupby("player")
    .apply(lambda g: g["balls_bowled"].sum() / g["matches"].sum())
    .reset_index(name="bowl_per_match")
)
regular_bowlers = bpm_career[bpm_career["bowl_per_match"] >= 6]["player"]
ar_pool = ar_pool[ar_pool["player"].isin(regular_bowlers)].copy()

# Add bowl_per_match as a column on each row so the dashboard can display it
ar_pool = ar_pool.merge(bpm_career, on="player")

print(f"After bowling regularity filter: {len(ar_pool)} player-seasons, {ar_pool['player'].nunique()} unique players")

# --- Step 6: Re-compute z-scores within the stricter pool ---
# The Step 4 z-scores were against the loose 50b+50b pool. After removing low-bpm
# players, re-normalize so each z-score reflects standing among genuine allrounders.
ar_pool["bat_z"] = ar_pool.groupby("season")["sr"].transform(
    lambda x: (x - x.mean()) / x.std() if x.std() > 0 else 0.0
)
ar_pool["bowl_z"] = ar_pool.groupby("season")["economy"].transform(
    lambda x: -(x - x.mean()) / x.std() if x.std() > 0 else 0.0
)
ar_pool["combined_z"] = ar_pool["bat_z"] + ar_pool["bowl_z"]

print("\nUpdated z-score distribution (stricter pool):")
print(ar_pool[["bat_z", "bowl_z", "combined_z"]].describe().round(3))

After bowling regularity filter: 376 player-seasons, 123 unique players

Updated z-score distribution (stricter pool):
        bat_z  bowl_z  combined_z
count 376.000 376.000     376.000
mean    0.000   0.000       0.000
std     0.976   0.976       1.331
min    -2.228  -2.605      -3.809
25%    -0.715  -0.635      -0.964
50%    -0.094   0.097       0.002
75%     0.610   0.777       0.876
max     2.906   2.350       3.642


In [7]:
# Save allrounder_scores.csv
out_cols = ['season', 'player', 'sr', 'economy', 'bat_z', 'bowl_z', 'combined_z',
            'balls_faced', 'balls_bowled', 'runs', 'runs_conceded', 'matches', 'bowl_per_match']
ar_pool[out_cols].sort_values(['season', 'combined_z'], ascending=[True, False]).to_csv(
    '../data/processed/allrounder_scores.csv', index=False
)
print(f'Saved {len(ar_pool)} rows to data/processed/allrounder_scores.csv')

Saved 376 rows to data/processed/allrounder_scores.csv


## 4. Season-Best Allrounder

For each season, identify the player with the highest combined_z. This is used by the dashboard's season-best chart on the Allrounders page.

In [8]:
season_best = (
    ar_pool.loc[ar_pool.groupby('season')['combined_z'].idxmax()]
    .sort_values('season')
    .reset_index(drop=True)
)[['season', 'player', 'combined_z', 'bat_z', 'bowl_z']]

print('Season-best allrounder per season:')
print(season_best.to_string(index=False))

season_best.to_csv('../data/processed/allrounder_season_best.csv', index=False)
print(f'\nSaved {len(season_best)} rows to data/processed/allrounder_season_best.csv')

Season-best allrounder per season:
 season          player  combined_z  bat_z  bowl_z
   2008     MF Maharoof       1.850  0.921   0.928
   2009 Harbhajan Singh       3.487  2.146   1.341
   2010      KA Pollard       2.737  1.996   0.741
   2011        CH Gayle       3.642  2.906   0.736
   2012        DR Smith       2.443  1.980   0.463
   2013 Harbhajan Singh       2.024  0.944   1.080
   2014 Shakib Al Hasan       1.942  0.737   1.205
   2015      AD Russell       2.999  2.511   0.488
   2016        CH Gayle       2.990  0.640   2.350
   2017      GJ Maxwell       2.913  1.504   1.408
   2018       K Gowtham       2.549  1.898   0.650
   2019          MM Ali       1.642  0.588   1.054
   2020       JC Archer       2.913  1.789   1.124
   2021      KA Pollard       1.792  1.143   0.649
   2022      GJ Maxwell       2.652  1.392   1.260
   2023     Rashid Khan       3.354  2.773   0.581
   2024       SP Narine       3.478  1.900   1.578
   2025       SP Narine       2.114  0.763   1.

## Summary

**Allrounder pool** (50+ balls faced AND 50+ balls bowled in same season):
- Covers all IPL seasons from 2008 to 2026
- Each player's batting SR and bowling economy is z-scored against that season's allrounder pool only
- This gives a fair comparison: Hardik's economy is measured against other allrounders, not against Bumrah

**Saved outputs**:
- `data/processed/allrounder_scores.csv` - per player per season z-scores
- `data/processed/allrounder_season_best.csv` - best combined-z allrounder per season